First we must load the dataset using Pandas -> this can use chunking later, with tqdm; actually this will depend on compatibility with torch dataset/dataloader

In [1]:
import pandas as pd

DATA_DIR = "kuairec/data/"

print("Loading big matrix...")
big_matrix = pd.read_csv(
    DATA_DIR + "big_matrix.csv",
    usecols=[
        'user_id',
        'video_id',
        'watch_ratio'
    ]
)
print("Big matrix loaded!")

Loading big matrix...
Big matrix loaded!


In [3]:
big_matrix

,user_id,video_id,watch_ratio
0,0,3649,1.273397
1,0,9598,1.244082
2,0,5262,0.107613
3,0,1963,0.089885
4,0,8234,0.078000
...,...,...,...
12530801,7175,1281,0.247241
12530802,7175,3407,0.576526
12530803,7175,10360,0.340597
12530804,7175,10360,0.913400


Start with the baseline of using a binary implicit like/dislike via a threshold

In [4]:
WATCH_LIKE_THRESHOLD = 2.0

big_matrix.watch_ratio = big_matrix.watch_ratio >= WATCH_LIKE_THRESHOLD
print("Binary threshold applied")

Binary threshold applied


In [5]:
big_matrix[big_matrix.watch_ratio]

,user_id,video_id,watch_ratio
7,0,6812,True
11,0,5274,True
12,0,179,True
17,0,171,True
23,0,211,True
...,...,...,...
12530735,7175,6502,True
12530749,7175,7951,True
12530759,7175,2970,True
12530767,7175,8898,True


In [6]:
big_matrix

,user_id,video_id,watch_ratio
0,0,3649,False
1,0,9598,False
2,0,5262,False
3,0,1963,False
4,0,8234,False
...,...,...,...
12530801,7175,1281,False
12530802,7175,3407,False
12530803,7175,10360,False
12530804,7175,10360,False


In [7]:
big_matrix['video_id'].unique()

array([3649, 9598, 5262, ..., 2915, 3246, 3899], shape=(10728,))

In [47]:
import torch
from torch import nn

USER_ID_DIMS = ITEM_ID_DIMS = 128

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()

        # by default, you would need to encode the IDs as one-hot vectors
        # these would be high-dimensional and extremely sparse; it is preferable
        # to learn a 128-width dense embedding vector instead to represent these IDs
        ## nn.Embedding does this efficiently
        self.user_id_embedder = nn.Embedding(7176, USER_ID_DIMS) # 7176 users total
        self.item_id_embedder = nn.Embedding(10728, ITEM_ID_DIMS) # 10728 items total
        
        self.user_tower = nn.Sequential(
            nn.Linear(USER_ID_DIMS, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

        self.item_tower = nn.Sequential(
            nn.Linear(ITEM_ID_DIMS, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, user_ids, item_ids):
        u = self.user_id_embedder(user_ids)
        i = self.item_id_embedder(item_ids)

        u = self.user_tower(u)
        i = self.item_tower(i)

        # calculate dot products between users and items
        # these are the raw scores (logits)
        logits = u @ i.T
        return logits

model = TwoTower().to(device)
print(model)

TwoTower(
  (user_id_embedder): Embedding(7176, 128)
  (item_id_embedder): Embedding(10728, 128)
  (user_tower): Sequential(
    (0): Linear(in_features=128, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=128, bias=True)
  )
  (item_tower): Sequential(
    (0): Linear(in_features=128, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=128, bias=True)
  )
)


In [9]:
x = torch.arange(4)
y = torch.arange(8)

logits = model(x, y)

logits

tensor([[-0.1926, -0.1225, -0.0900, -0.0256, -0.2110,  0.0055, -0.2227, -0.1073],
        [-0.0462, -0.0519,  0.0428, -0.1190, -0.0904,  0.0078, -0.1107,  0.0107],
        [-0.1204, -0.1337,  0.0511, -0.0889, -0.2131, -0.0661, -0.1609, -0.1550],
        [-0.0415, -0.0434, -0.0407, -0.1238, -0.0813, -0.1078, -0.1468, -0.0147]],
       grad_fn=<MmBackward0>)

In [10]:
nn.Softmax(dim=1)(logits)

tensor([[0.1160, 0.1244, 0.1285, 0.1371, 0.1139, 0.1414, 0.1125, 0.1263],
        [0.1246, 0.1239, 0.1362, 0.1159, 0.1192, 0.1315, 0.1168, 0.1319],
        [0.1235, 0.1218, 0.1466, 0.1274, 0.1125, 0.1304, 0.1186, 0.1193],
        [0.1291, 0.1289, 0.1292, 0.1189, 0.1241, 0.1209, 0.1162, 0.1326]],
       grad_fn=<SoftmaxBackward0>)

In [11]:
# max gives you both the values (predicted probabilities)
# and the indices (the column no. i.e. the class/ predicted item number)
nn.Softmax(dim=1)(logits).argmax(1)

tensor([5, 2, 2, 7])

In [12]:
loss = nn.CrossEntropyLoss()

print(x)
print(y)
print(logits)
print(nn.Softmax(dim=1)(logits))
print(nn.Softmax(dim=1)(logits).argmax(1))

tensor([0, 1, 2, 3])
tensor([0, 1, 2, 3, 4, 5, 6, 7])
tensor([[-0.1926, -0.1225, -0.0900, -0.0256, -0.2110,  0.0055, -0.2227, -0.1073],
        [-0.0462, -0.0519,  0.0428, -0.1190, -0.0904,  0.0078, -0.1107,  0.0107],
        [-0.1204, -0.1337,  0.0511, -0.0889, -0.2131, -0.0661, -0.1609, -0.1550],
        [-0.0415, -0.0434, -0.0407, -0.1238, -0.0813, -0.1078, -0.1468, -0.0147]],
       grad_fn=<MmBackward0>)
tensor([[0.1160, 0.1244, 0.1285, 0.1371, 0.1139, 0.1414, 0.1125, 0.1263],
        [0.1246, 0.1239, 0.1362, 0.1159, 0.1192, 0.1315, 0.1168, 0.1319],
        [0.1235, 0.1218, 0.1466, 0.1274, 0.1125, 0.1304, 0.1186, 0.1193],
        [0.1291, 0.1289, 0.1292, 0.1189, 0.1241, 0.1209, 0.1162, 0.1326]],
       grad_fn=<SoftmaxBackward0>)
tensor([5, 2, 2, 7])


In [13]:
target = torch.tensor([
    [0, 0, 1, 0, 0, 0, 0, 0], # i.e. user 0 likes film 2
    [0, 0, 0, 0, 0, 0, 0, 1], # user 1 -> film 8
    [0, 0, 0, 0, 1, 0, 0, 0], # user 2 -> film 4
    [1, 0, 0, 0, 0, 0, 0, 0]  # user 3 -> film 1
]).float().to(device)

print(target)

tensor([[0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.]])


In [14]:
# note loss with itself is not 0 (but can approach 0 if I scale it from 0s and 1s
# to e.g. 0s and 100s because then the softmax will approach 0 probability for the 0 entries
# - just how softmax works since 0 and 1 don't differ too much in value)
loss(target, target)

tensor(1.2740)

In [15]:
opt = torch.optim.SGD(model.parameters(), lr=1e-3)

opt

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [16]:
loss

CrossEntropyLoss()

In [17]:
loss(logits, target)

tensor(2.0772, grad_fn=<DivBackward1>)

In [18]:
model.train()
pred = model(x, y)
l = loss(pred, target)
l.backward()
opt.step()
opt.zero_grad()
print(nn.Softmax(dim=1)(pred))
print(target)
print(l)

tensor([[0.1160, 0.1244, 0.1285, 0.1371, 0.1139, 0.1414, 0.1125, 0.1263],
        [0.1246, 0.1239, 0.1362, 0.1159, 0.1192, 0.1315, 0.1168, 0.1319],
        [0.1235, 0.1218, 0.1466, 0.1274, 0.1125, 0.1304, 0.1186, 0.1193],
        [0.1291, 0.1289, 0.1292, 0.1189, 0.1241, 0.1209, 0.1162, 0.1326]],
       grad_fn=<SoftmaxBackward0>)
tensor([[0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.]])
tensor(2.0772, grad_fn=<DivBackward1>)


In [2]:
big_matrix

,user_id,video_id,watch_ratio
0,0,3649,1.273397
1,0,9598,1.244082
2,0,5262,0.107613
3,0,1963,0.089885
4,0,8234,0.078000
...,...,...,...
12530801,7175,1281,0.247241
12530802,7175,3407,0.576526
12530803,7175,10360,0.340597
12530804,7175,10360,0.913400


In [18]:
tuple(big_matrix.iloc[123])

(0.0, 8119.0, 0.0811304431414015)

In [19]:
from torch.utils.data import Dataset

class KRBig(Dataset):
    def __init__(self, krbig_table):
        self.data = krbig_table

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return tuple(big_matrix.iloc[idx])

In [23]:
from torch.utils.data import DataLoader

krbig = KRBig(big_matrix)

ldr_krbig = DataLoader(krbig, batch_size=64, shuffle=True)

In [28]:
ldr_krbig.__dict__

{'dataset': <__main__.KRBig at 0x7b66b4713020>,
 'num_workers': 0,
 'prefetch_factor': None,
 'pin_memory': False,
 'pin_memory_device': '',
 'timeout': 0,
 'worker_init_fn': None,
 '_DataLoader__multiprocessing_context': None,
 'in_order': True,
 '_dataset_kind': 0,
 'batch_size': 64,
 'drop_last': False,
 'sampler': <torch.utils.data.sampler.RandomSampler at 0x7b66b4713740>,
 'batch_sampler': <torch.utils.data.sampler.BatchSampler at 0x7b66b47130e0>,
 'generator': None,
 'collate_fn': <function torch.utils.data._utils.collate.default_collate(batch)>,
 'persistent_workers': False,
 '_DataLoader__initialized': True,
 '_IterableDataset_len_called': None,
 '_iterator': None}

In [31]:
dir(ldr_krbig)

['_DataLoader__initialized',
 '_DataLoader__multiprocessing_context',
 '_IterableDataset_len_called',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_auto_collation',
 '_dataset_kind',
 '_get_iterator',
 '_index_sampler',
 '_iterator',
 'batch_sampler',
 'batch_size',
 'check_worker_number_rationality',
 'collate_fn',
 'dataset',
 'drop_last',
 'generator',
 'in_order',
 'multiprocessing_context',
 'num_workers',
 'persistent_workers',
 'pin_memory',
 'pin_memory_device',
 'prefetch_factor',
 'sampler',
 'timeout',
 'worker_init_fn']

In [33]:
# it = iter(ldr_krbig)
# looks like iter is just a wrapper so it's O(1) - iterables in python usually are wrappers

next(iter(ldr_krbig))

[tensor([1468., 5038.,  498.,   67., 4775.,  525., 6234., 3801., 5604., 4942.,
         4894., 7009., 1680., 5363., 3988., 6521., 5435., 2774., 5708., 6966.,
         2473., 6804., 2688., 4099., 3487., 4811., 2102., 4432., 1997.,  296.,
         6489., 4938., 2164.,  153., 3586., 1555., 4079., 3121., 6934., 5406.,
          837., 4225., 1090., 3158., 1735., 5661., 3420., 5768., 6799., 7088.,
         5216., 4985., 6472., 1519., 5761., 1599., 6250., 3697., 2961., 4239.,
         2381., 6348., 3141., 3181.], dtype=torch.float64),
 tensor([ 2894.,  7736.,  8477.,  1432.,  8977.,  1184.,  7748.,  2065., 10663.,
         10332.,  7285.,   288.,  3299.,  2315.,  6173.,  9678.,  7267.,  1333.,
          2137.,  2289.,   712.,   726.,  7737.,  4791.,  5703.,  4121.,  8586.,
          2436.,  2160.,  5782.,  9848.,  4410.,  2592.,  7078.,  8976.,  6834.,
          2971.,   132.,  2713.,  6942., 10191.,  4655.,  2698.,   695., 10346.,
          6258.,  9074.,  2994.,  1280.,   932.,  2914.,  527

In [37]:
# In reality I'm going to narrow this down to just positive samples (for strong signals)
# And use in-batch negatives

big_matrix[big_matrix.watch_ratio >= 2.0].iloc[:, 0:2]

,user_id,video_id
7,0,6812
11,0,5274
12,0,179
17,0,171
23,0,211
...,...,...
12530735,7175,6502
12530749,7175,7951
12530759,7175,2970
12530767,7175,8898


In [83]:
class KRBig2(Dataset):
    def __init__(self, data):
        self.data = data[data.watch_ratio >= 2.0].iloc[:, 0:2]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return tuple(self.data.iloc[idx])

krbig = KRBig2(big_matrix)

# for now i'll use the default dataloader; assume that it's returning unique users
# i.e. each user is optimising for a single item in the batch without collisions
it_krbig = iter(DataLoader(krbig, batch_size=512, shuffle=True))

next(it_krbig)

[tensor([4147, 5373, 2404,  694, 4258, 3473,  714, 6416,  979, 2783, 3295, 4394,
         1716, 1866, 2304, 5549, 4858, 3893, 5350, 5723, 4148,  408, 3705, 4992,
         5992, 6729, 5380, 1440, 6193, 1599, 4779, 4880,  899, 4762, 1548, 1633,
         4681, 6288,    5, 4944, 4734, 4182,  213, 5168, 5074, 6002, 3138, 2465,
         1841, 5031, 4458, 2190, 5293, 1767, 6713, 6810, 4142, 1614, 6912, 1215,
          933,  643, 1526, 5176, 1443, 5638, 6903, 2119, 5084,  837,   85, 4097,
          126, 2975, 3337,  119, 1392, 1212, 5057, 5194, 2487,  713, 6940, 3217,
         6696,  201, 3424, 3317, 5274, 6042, 6444, 2199, 3139, 2486, 4029, 1162,
         6281, 6966, 1503, 6137, 4846, 2774, 4697, 3249, 5481, 4168, 2745, 1039,
         6531, 1985, 3444, 1777, 5819, 3200, 6348, 1478, 2030, 5243,  238, 5851,
         2571, 2340, 4136, 6687, 4713, 6316, 1409, 5255,  862,  331, 6874, 2987,
         5242,  117, 6898, 5684, 3287, 4203, 3865, 1833, 6985, 1199,  392, 4842,
         3329,  483, 5991, 3

In [84]:
next(it_krbig)

[tensor([1379, 4585, 3876, 6300, 1476, 4259, 6385, 4827, 1051, 4503, 1870, 4170,
         3884, 4534,  147, 5561, 4024,  576, 5780, 1783, 7099, 1264, 1035, 5383,
         4627, 5130, 5081, 3938,  422, 6173, 4035, 2554, 2942, 7010, 4036, 4880,
          470, 1573, 4582, 5983, 6351,   68, 7114, 6165, 4959, 1082,  913, 3696,
         5204, 5373, 4895, 6748, 6128, 1706, 3525, 4549,  408, 6021, 3444, 1354,
          895, 5592, 4951, 5732, 4728, 5984, 1180, 2021, 5536, 6257,  499, 4521,
         3542, 6031, 5445, 5899, 6623, 5261, 4152, 2531, 7169, 4217, 2223, 6756,
          225, 4129, 5238, 4002, 1429, 5945, 4616, 1925, 3922,  913, 4833, 3370,
         1307, 5529, 2821, 3407, 2740, 3033, 6713,   35, 5523, 4696, 4519, 3056,
         4851, 1164, 3945, 2792, 6160,  358,  188, 4895,   11, 5504, 3366, 5988,
         6879, 1239,  164,  516, 6323, 3327, 5905,  264, 1569, 1259, 1852, 1536,
         5119, 1092, 2701, 4921, 3585, 3532, 5409, 4389, 6640, 4870, 2774, 4933,
         3358, 1724, 2067, 1

In [85]:
# Now I'll need to prepare the training target
# Optimise for the item that it likes

batch = next(it_krbig)

res = model(batch[0], batch[1])

res

tensor([[-0.2094, -0.2506, -0.2603,  ..., -0.1914, -0.2022, -0.2206],
        [-0.0730, -0.1662, -0.1114,  ..., -0.0185,  0.0312,  0.0060],
        [ 0.0294, -0.1020, -0.1142,  ...,  0.0215, -0.0655, -0.0907],
        ...,
        [-0.0834, -0.0934, -0.1680,  ..., -0.1151, -0.0274, -0.0307],
        [-0.1953, -0.1994, -0.1991,  ..., -0.1537, -0.0717, -0.1929],
        [-0.0882, -0.0041, -0.0615,  ..., -0.0281,  0.1127, -0.0173]],
       grad_fn=<MmBackward0>)

In [86]:
# Now note that since users & items are passed s.t. that each user
# likes the item of the same index/pos in the item list (since just
# user-item interactions) then it's essentially a diagonal matrix to
# aim for here

print(res.shape)

nn.Softmax(dim=1)(res).max(1)

torch.Size([512, 512])


torch.return_types.max(
values=tensor([0.0023, 0.0023, 0.0023, 0.0023, 0.0024, 0.0024, 0.0023, 0.0023, 0.0024,
        0.0024, 0.0023, 0.0023, 0.0025, 0.0023, 0.0023, 0.0024, 0.0025, 0.0024,
        0.0023, 0.0023, 0.0023, 0.0024, 0.0024, 0.0024, 0.0023, 0.0023, 0.0023,
        0.0023, 0.0024, 0.0024, 0.0023, 0.0024, 0.0023, 0.0023, 0.0023, 0.0023,
        0.0024, 0.0025, 0.0024, 0.0023, 0.0024, 0.0023, 0.0023, 0.0023, 0.0023,
        0.0023, 0.0025, 0.0022, 0.0024, 0.0024, 0.0023, 0.0024, 0.0024, 0.0023,
        0.0024, 0.0023, 0.0023, 0.0023, 0.0025, 0.0025, 0.0024, 0.0023, 0.0024,
        0.0023, 0.0024, 0.0023, 0.0023, 0.0023, 0.0023, 0.0024, 0.0023, 0.0026,
        0.0023, 0.0024, 0.0024, 0.0023, 0.0025, 0.0022, 0.0025, 0.0024, 0.0023,
        0.0023, 0.0023, 0.0024, 0.0023, 0.0024, 0.0024, 0.0024, 0.0024, 0.0023,
        0.0024, 0.0024, 0.0023, 0.0023, 0.0024, 0.0024, 0.0023, 0.0024, 0.0023,
        0.0023, 0.0023, 0.0025, 0.0025, 0.0023, 0.0023, 0.0024, 0.0026, 0.0024,
        0

In [87]:
# this should aim to be [0, 1, 2, 3, ...] etc on argmax since diagonal
loss_fn = nn.CrossEntropyLoss()

target = torch.eye(512)
target

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]])

In [88]:
loss_fn(res, target)

tensor(6.2435, grad_fn=<DivBackward1>)

In [90]:
torch.eye(512)

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]])

In [94]:
target = torch.eye(512)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()
for i in range(1000):
    batch = next(it_krbig)
    logits = model(batch[0], batch[1])
    loss = loss_fn(logits, target)
    if i%100 == 0:
        print(f"Iteration {i} loss: {loss}")

    loss.backward()
    opt.step()
    opt.zero_grad()

Iteration 0 loss: 6.226714611053467
Iteration 100 loss: 6.217182159423828
Iteration 200 loss: 6.210747241973877
Iteration 300 loss: 6.1824188232421875
Iteration 400 loss: 6.158812522888184
Iteration 500 loss: 6.150500297546387
Iteration 600 loss: 6.157151222229004


ValueError: Expected input batch_size (120) to match target batch_size (512).

In [111]:
# Convergence seems to take a while - it is a very large dataset to be fair
# Sanity check: can it converge to a known fixed dataset quickly
krbig = KRBig2(big_matrix)

it = iter(DataLoader(krbig, batch_size=128, shuffle=True))
batch = next(it)

model = TwoTower().to(device)
model.train()

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

target = torch.eye(128)

loss_fn(model(batch[0], batch[1]), target)

for i in range(100):
    opt.zero_grad()

    #batch = next(it)
    
    loss = loss_fn(model(batch[0], batch[1]), target)
    print(f"Iteration {i} loss: {loss}")

    loss.backward()
    opt.step()

Iteration 0 loss: 4.853856563568115
Iteration 1 loss: 4.417910099029541
Iteration 2 loss: 3.743466377258301
Iteration 3 loss: 2.713430166244507
Iteration 4 loss: 1.5082589387893677
Iteration 5 loss: 0.5853583812713623
Iteration 6 loss: 0.2131100594997406
Iteration 7 loss: 0.17292441427707672
Iteration 8 loss: 0.21490047872066498
Iteration 9 loss: 0.1965709775686264
Iteration 10 loss: 0.27440983057022095
Iteration 11 loss: 0.16613435745239258
Iteration 12 loss: 0.20126068592071533
Iteration 13 loss: 0.17885085940361023
Iteration 14 loss: 0.14486262202262878
Iteration 15 loss: 0.11940787732601166
Iteration 16 loss: 0.12148920446634293
Iteration 17 loss: 0.12415669858455658
Iteration 18 loss: 0.14129094779491425
Iteration 19 loss: 0.16769303381443024
Iteration 20 loss: 0.11057920753955841
Iteration 21 loss: 0.1845914125442505
Iteration 22 loss: 0.12300438433885574
Iteration 23 loss: 0.22459016740322113
Iteration 24 loss: 0.22556009888648987
Iteration 25 loss: 0.15155334770679474
Iteration